# Python w analizie danych – co z "czystego" Pythona przydaje się najczęściej

Krótki przegląd wzorców, które wracają w każdej analizie danych. Zakładamy, że znasz już materiał z notatnika *Podstawy Python*. Tu skupiamy się na tym, **jak tych elementów używa się w pracy z danymi** – zanim przejdziemy do NumPy i Pandas.

## 1. Dane jako lista słowników

Najczęstszy format danych "z zewnątrz" (JSON, odpowiedzi API, wiersze z bazy) to **lista słowników** – każdy słownik to jeden rekord. Tak samo wygląda to po wczytaniu pliku JSON i tak łatwo zamienić to na tabelę w Pandas.

In [1]:
sprzedaz = [
    {"miasto": "Warszawa", "produkt": "laptop", "ilosc": 3, "cena": 3200.0},
    {"miasto": "Kraków", "produkt": "monitor", "ilosc": 5, "cena": 750.0},
    {"miasto": "Warszawa", "produkt": "monitor", "ilosc": 2, "cena": 780.0},
    {"miasto": "Gdańsk", "produkt": "laptop", "ilosc": 1, "cena": 3100.0},
    {"miasto": "Kraków", "produkt": "laptop", "ilosc": 4, "cena": 3250.0},
]

print(sprzedaz[0])
print(sprzedaz[0]["cena"])

{'miasto': 'Warszawa', 'produkt': 'laptop', 'ilosc': 3, 'cena': 3200.0}
3200.0


## 2. Comprehensions i `sorted(key=lambda ...)` – filtrowanie i transformacja

In [2]:
# filtrowanie rekordów
duze_zamowienia = [r for r in sprzedaz if r["ilosc"] >= 3]
print(len(duze_zamowienia), "duże zamówienia")

# nowa wartość wyliczona z każdego rekordu
przychody = [r["ilosc"] * r["cena"] for r in sprzedaz]
print(przychody, "-> suma:", sum(przychody))

# sortowanie po wybranym polu
najdrozsze = sorted(sprzedaz, key=lambda r: r["cena"], reverse=True)[:2]
print([(r["produkt"], r["cena"]) for r in najdrozsze])

3 duże zamówienia
[9600.0, 3750.0, 1560.0, 3100.0, 13000.0] -> suma: 31010.0
[('laptop', 3250.0), ('laptop', 3200.0)]


## 3. Zliczanie i grupowanie: `Counter`, `defaultdict`

In [ ]:
from collections import Counter, defaultdict

# ile razy występuje dana wartość
print(Counter(r["miasto"] for r in sprzedaz))
print(Counter(r["produkt"] for r in sprzedaz).most_common(1))

# sumowanie wg grupy
przychod_wg_miasta = defaultdict(float)
for r in sprzedaz:
    przychod_wg_miasta[r["miasto"]] += r["ilosc"] * r["cena"]

dict(przychod_wg_miasta)

Counter({'Warszawa': 2, 'Kraków': 2, 'Gdańsk': 1})
[('laptop', 3)]


{'Warszawa': 11160.0, 'Kraków': 16750.0, 'Gdańsk': 3100.0}

## 4. `enumerate` i `zip` – praca na kilku listach naraz

In [5]:
miasta = ["Warszawa", "Kraków", "Gdańsk"]
wyniki = [12400, 15750, 3100]

for i, (miasto, wynik) in enumerate(zip(miasta, wyniki), start=1):
    print(f"{i}. {miasto}: {wynik}")

dict(zip(miasta, wyniki))  # dwie listy -> słownik

1. Warszawa: 12400
2. Kraków: 15750
3. Gdańsk: 3100


{'Warszawa': 12400, 'Kraków': 15750, 'Gdańsk': 3100}

## 5. f-stringi – czytelne wyniki i raporty

Format `:,.2f` (separator tysięcy i 2 miejsca po przecinku) oraz `:.1%` (procenty) przydają się przy prezentacji wyników.

In [6]:
razem = sum(przychody)
udzial_laptopow = sum(r["ilosc"] * r["cena"] for r in sprzedaz if r["produkt"] == "laptop") / razem

print(f"Przychód: {razem:,.2f} zł")
print(f"Udział laptopów: {udzial_laptopow:.1%}")

Przychód: 31,010.00 zł
Udział laptopów: 82.9%


## 6. Brudne dane i obsługa wyjątków

Dane z życia bywają niespójne, np. liczby z przecinkiem zamiast kropki. `try/except` pozwala nie przerywać całej analizy przez jedną złą wartość.

In [9]:
surowe = ["12.5", "7,8", "brak", "10"]

oczyszczone = []
for tekst in surowe:
    try:
        oczyszczone.append(float(tekst.replace(",", ".")))
    except ValueError:
        print(f"Pomijam niepoprawną wartość: {tekst!r}")

oczyszczone

Pomijam niepoprawną wartość: 'brak'


[12.5, 7.8, 10.0]

## 7. Daty i czas: `datetime`

In [11]:
from datetime import datetime, timedelta

data = datetime.strptime("2026-03-15", "%Y-%m-%d")
print(data.year, data.month, data.strftime("%A"))
print(data + timedelta(days=30))
print((datetime(2026, 12, 31) - data).days, "dni do końca roku")

2026 3 Sunday
2026-04-14 00:00:00
291 dni do końca roku


## 8. Proste statystyki: `statistics`, `math`

In [12]:
import math
import statistics

dane = [12, 15, 11, 18, 14, 90]  # jedna wartość odstająca

print("średnia :", statistics.mean(dane))
print("mediana :", statistics.median(dane))
print("odch.std:", round(statistics.stdev(dane), 2))
print("pierwiastek z 16:", math.sqrt(16))

średnia : 26.666666666666668
mediana : 14.5
odch.std: 31.12
pierwiastek z 16: 4.0


Średnia jest wrażliwa na wartości odstające (tu 90), a mediana – nie. To jedna z podstawowych obserwacji w EDA.

## 9. Co dalej? Python vs Pandas

Poniższy przykład z punktu 3 (sumowanie przychodu wg miasta) w Pandas to jedna linia:

In [13]:
import pandas as pd

df = pd.DataFrame(sprzedaz)  # lista słowników -> tabela
df["przychod"] = df["ilosc"] * df["cena"]
df.groupby("miasto")["przychod"].sum()

miasto
Gdańsk       3100.0
Kraków      16750.0
Warszawa    11160.0
Name: przychod, dtype: float64